# 中文规范化流水线（fork 增强）

`semantica-cn` 的中文规范化能力：**NFKC 全角转半角 → 删 CJK 噪声空格 → 中文日期解析**。纯标准库实现。文档见「中文规范化」参考页（docs_zh/reference/zh-normalization.md）。

In [ ]:
import unicodedata
from datetime import datetime

from semantica.normalize import CJKSpacingNormalizer, ZhDateParser, methods

## 1. NFKC + CJK 空格清理

PDF 抽取后的中文文本常带噪声空格。`policy="remove"` 一次清干净中英文边界与 CJK 相邻空格，幂等可重入：

Markdown 语法完整保留（默认开，`preserve_markdown=True`）：标题/引用/列表标记后的空格、代码块与行内代码内容、强调结构都不动——`## 折扣政策` 不会被吃成 `##折扣政策`，下游的标题切分与定位不受影响。纯文本场景可设 `preserve_markdown=False`。

In [ ]:
raw = "支 付 $5M，于 2026年10月2日 到 账"

t = unicodedata.normalize("NFKC", raw)
t = CJKSpacingNormalizer(policy="remove").normalize(t)
print(t)  # -> 支付$5M,于2026年10月2日到账（NFKC 顺带把全角标点转半角）


`policy="add"`（默认）反向操作——中英文边界加空格（pangu 风格），适合面向人阅读的输出：

In [ ]:
CJKSpacingNormalizer().normalize("Semantica知识图谱，AI驱动")
# -> 'Semantica 知识图谱，AI 驱动'

## 2. 中文日期解析

家族覆盖：完整日期、带时间、汉字数字（二〇二六年十月二日）、无年份、**区间**、仅年份、相对词（去年/明天/下周X）、锚定数量偏移（前三个自然月）。

In [ ]:
p = ZhDateParser()
print(p.parse("2026年10月2日 14:30")["value"])
rng = p.parse("2025年1月1日至2025年12月31日")
print(rng["value"], "~", rng["value_end"])
print(p.parse("二〇二五年十二月三十一日")["value"])
print(p.parse("前三个自然月", reference=datetime(2025, 1, 1))["value"])

失败语义不抛异常：非法日期（2月30日）拒绝；农历（腊月初八）原样保留并计数：

In [ ]:
print(p.parse("2月30日"))      # value=None（拒绝，不溢出成 3月2日）
print(p.parse("腊月初八"))    # unsupported_matched 保留原文
print(p.unsupported_count)     # 计数

## 3. 方法分发入口

`method="cjk_spacing"` / `method="cn_date"` 直接走 methods 层；需要 `DateNormalizer` 直取中文时用 fork 扩展类 `ZhDateNormalizer`（中文命中走解析器，未命中回退父类 dateutil）：

In [ ]:
print(methods.normalize_text("知识 图谱", method="cjk_spacing", policy="remove"))
print(methods.normalize_date("2026年10月2日", method="cn_date", format="date"))

from semantica.normalize import ZhDateNormalizer
print(ZhDateNormalizer().normalize_date("2026年10月2日"))  # -> 2026-10-02T...
print(ZhDateNormalizer().normalize_date("2023-01-15"))   # 回退父类，行为不变